In [33]:
import torch
import torch.nn as nn

In [34]:
class GCNLayer(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.lin = nn.Linear(n_in, n_out)
        self.relu = nn.ReLU()
        

    def forward(self, x, A_norm):
        x = A_norm @ x
        x = self.lin(x)
        x = self.relu(x)
        return x
                

In [35]:
class FireGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.g1 = GCNLayer(1,16)
        self.g2 = GCNLayer(16,16)
        self.head = nn.Linear(16,2)

    def forward(self, x, A_norm):
        x = self.g1(x, A_norm)
        x = self.g2(x, A_norm)
        x = self.head(x)
        return x

In [36]:
def build_line_adjacency(n_cells):     
    A = torch.eye(20)
    for i in range (19):
        A[i, i+1] = 1
        A[i+1, i] = 1
    deg = A.sum(dim=1, keepdim=True)
    A_norm = A / deg
    return A, A_norm

In [37]:
def make_fire_data(A, n_states = 500, p = 0.15):
    X = (torch.rand(500, 20, 1) < 0.15).float()
    y = (A @ X > 0).long().squeeze(-1)
    return X, y

In [38]:
def train(model, X, y, A_norm, loss_fn, optimizer, steps = 300):

    for step in range(steps):
        out = model(X, A_norm)
        loss = loss_fn(out.reshape(-1, 2), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(step, loss.item()) 
     
        
    

In [39]:
def evaluate(model, X, y, A_norm):
    with torch.no_grad():
        out = model(X, A_norm)
        predictions = out.argmax(dim=-1)
        correct = (predictions == y).sum().item()
    return round(correct / y.numel(), 4)
        
    

In [40]:
A, A_norm = build_line_adjacency(20)
X, y = make_fire_data(A)
model = FireGNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
train(model, X[:400], y[:400], A_norm, nn.CrossEntropyLoss(), optimizer)
print(evaluate(model, X[400:], y[400:], A_norm))

for i in range(5):
    now  = "".join(str(int(v)) for v in X[i].flatten())
    nxt  = "".join(str(int(v)) for v in y[i].flatten())
    print(f"now : {now}")
    print(f"next: {nxt}\n")

0 0.6725233197212219
50 0.06119745597243309
100 0.04393120855093002
150 0.04207373037934303
200 0.04137563332915306
250 0.04103821516036987
0.9945
now : 00010000010100000000
next: 00111000111110000000

now : 00000000001000100010
next: 00000000011101110111

now : 00010100000010001000
next: 00111110000111011100

now : 00000000000110000000
next: 00000000001111000000

now : 00011000000000000000
next: 00111100000000000000

